# Evaluation Complete du Modele LoRA SDXL

Evaluation du modele **Stable Diffusion XL + LoRA** entraine sur CelebA.

**Etapes :**
1. Chargement SDXL + adaptateur LoRA depuis le checkpoint final
2. Generation de 20 images de test (prompts varies, seeds aleatoires)
3. CLIP Score — alignement texte-image
4. FID Score — qualite et realisme vs images reelles CelebA
5. Dashboard de visualisation + rapport JSON

**Checkpoint utilise :**
`/kaggle/input/datasets/souissiyoussef/diffusion-text2image/output/results/checkpoints/final`

## 1. Setup et imports

In [ ]:
# Installer le package CLIP d'OpenAI (absent par defaut sur Kaggle)
import subprocess
result = subprocess.run(
    ['pip', 'install', '--quiet', 'git+https://github.com/openai/CLIP.git'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('OK CLIP installe')
else:
    print('Erreur installation CLIP:', result.stderr[-200:])

In [ ]:
import sys, os, warnings, hashlib, time
warnings.filterwarnings('ignore')
from pathlib import Path
import torch
import numpy as np
from PIL import Image
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from datetime import datetime
import shutil

# ── Chemins canoniques Kaggle ─────────────────────────────────────────
KAGGLE_DATASET_ROOT  = Path('/kaggle/input/datasets/souissiyoussef/diffusion-text2image')
KAGGLE_RESULTS_ROOT  = KAGGLE_DATASET_ROOT / 'output' / 'results'
KAGGLE_CHECKPOINTS_DIR = KAGGLE_RESULTS_ROOT / 'checkpoints'

MODEL_ID    = 'stabilityai/stable-diffusion-xl-base-1.0'
VAE_MODEL_ID = 'madebyollin/sdxl-vae-fp16-fix'

WORKING_DIR = Path('/kaggle/working')
SAMPLES_DIR = WORKING_DIR / 'evaluation_samples'
SAMPLES_DIR.mkdir(exist_ok=True)

# ── Selectionner le meilleur checkpoint disponible ────────────────────
# Lit checkpoint_meta.json dans chaque checkpoint et choisit
# celui avec la best_loss la plus basse DANS LA PLAGE SAINE (> 0.01).
# En dessous de 0.01 = surapprentissage certain.
HEALTHY_LOSS_MIN = 0.01   # en dessous = overfit

def select_best_checkpoint(checkpoints_dir: Path) -> Path:
    best_path = None
    best_loss = float('inf')

    for ckpt_dir in sorted(checkpoints_dir.iterdir()):
        if not ckpt_dir.is_dir():
            continue
        meta_file = ckpt_dir / 'checkpoint_meta.json'
        if not meta_file.exists():
            continue
        with open(meta_file) as f:
            meta = json.load(f)
        loss = meta.get('best_loss', float('inf'))
        step = meta.get('global_step', 0)
        healthy = loss >= HEALTHY_LOSS_MIN
        print(f'  {ckpt_dir.name:25s} | step={step:6d} | best_loss={loss:.6f} | {"OK" if healthy else "OVERFIT — ignore"}')
        if healthy and loss < best_loss:
            best_loss = loss
            best_path = ckpt_dir

    return best_path, best_loss

print('Recherche du meilleur checkpoint...')
best_ckpt, best_loss_val = select_best_checkpoint(KAGGLE_CHECKPOINTS_DIR)

if best_ckpt is None:
    raise RuntimeError('Aucun checkpoint sain trouve dans ' + str(KAGGLE_CHECKPOINTS_DIR))

print(f'\nMeilleur checkpoint : {best_ckpt.name} (best_loss={best_loss_val:.6f})')

# Copier vers /kaggle/working/results/checkpoints/final
# pour que les scripts futurs utilisent toujours "final"
FINAL_DIR = WORKING_DIR / 'results' / 'checkpoints' / 'final'
FINAL_DIR.parent.mkdir(parents=True, exist_ok=True)
if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)
shutil.copytree(best_ckpt, FINAL_DIR)
print(f'Copie vers          : {FINAL_DIR}')

CHECKPOINT_PATH = FINAL_DIR

# ── Localiser les images CelebA reelles pour FID ─────────────────────
# Chemin exact : jessicali9530/celeba-dataset/img_align_celeba/img_align_celeba/
def find_celeba_images(max_images: int = 500):
    import csv, random

    CELEBA_ROOT = Path('/kaggle/input/datasets/jessicali9530/celeba-dataset')
    celeba_img_dir = CELEBA_ROOT / 'img_align_celeba' / 'img_align_celeba'
    if not celeba_img_dir.exists():
        celeba_img_dir = CELEBA_ROOT / 'img_align_celeba'
    if not celeba_img_dir.exists() or not any(celeba_img_dir.glob('*.jpg')):
        print(f'  AVERTISSEMENT : images CelebA introuvables a {CELEBA_ROOT}')
        return []

    total = sum(1 for _ in celeba_img_dir.glob('*.jpg'))
    print(f'  Images CelebA : {celeba_img_dir} ({total} fichiers)')

    # Utiliser val/metadata.csv pour prendre uniquement des images de validation
    val_csv = KAGGLE_DATASET_ROOT / 'data' / 'processed' / 'val' / 'metadata.csv'
    filenames = []
    if val_csv.exists():
        with open(val_csv) as f:
            reader = csv.DictReader(f)
            for row in reader:
                fname = Path(row.get('image_path', row.get('filename', ''))).name
                if fname:
                    filenames.append(fname)
        random.shuffle(filenames)
        print(f'  Val metadata  : {len(filenames)} entrees')
    else:
        filenames = [p.name for p in sorted(celeba_img_dir.glob('*.jpg'))]

    paths = []
    for fname in filenames:
        p = celeba_img_dir / fname
        if p.exists():
            paths.append(p)
        if len(paths) >= max_images:
            break

    print(f'  Images val resolues : {len(paths)} / {max_images}')
    return paths

# Ajouter src/ au path
src_dir = KAGGLE_DATASET_ROOT / 'src'
if src_dir.exists() and str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

print()
print('Setup complet')
print(f'Checkpoint utilise : {CHECKPOINT_PATH}')
print(f'Existe             : {CHECKPOINT_PATH.exists()}')
print(f'GPU                : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN"}')
print()
print('Recherche images CelebA reelles...')
real_celeba_paths = find_celeba_images(500)
if not real_celeba_paths:
    print('  FID ne sera pas calcule (images CelebA non trouvees).')
else:
    print(f'  OK : {len(real_celeba_paths)} images prets pour FID')


## 2. Chargement SDXL + LoRA

Chargement du pipeline SDXL avec le VAE stabilise et l'adaptateur LoRA entraine.

In [ ]:
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from peft import PeftModel

print('=' * 70)
print('CHARGEMENT DU MODELE SDXL + LORA')
print('=' * 70)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f'Checkpoint non trouve: {CHECKPOINT_PATH}')

# VAE stabilise (evite NaN en fp16 avec le VAE SDXL original)
print(f'\n1. VAE stabilise: {VAE_MODEL_ID}')
vae = AutoencoderKL.from_pretrained(VAE_MODEL_ID, torch_dtype=torch.float16)
print('   OK')

# Modele de base SDXL
print(f'\n2. Modele de base: {MODEL_ID}')
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    vae=vae,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
print('   OK')

# Adaptateur LoRA — charger sur UNet brut avec is_trainable=False (inference)
print(f'\n3. Adaptateur LoRA: {CHECKPOINT_PATH.name}')
pipe.unet = PeftModel.from_pretrained(
    pipe.unet,
    str(CHECKPOINT_PATH),
    adapter_name='default',
    is_trainable=False,
)
print('   OK')

pipe.to('cuda')
pipe.enable_attention_slicing()

print('\nLoRA actif :', hasattr(pipe.unet, 'peft_config'))
print('=' * 70)

## 3. Generation d'images de test

20 prompts varies avec seeds aleatoires (diversite garantie).
- `guidance_scale=9.0` : meilleur equilibre qualite/fidelite pour portraits
- `num_inference_steps=50` : qualite optimale
- `height=width=1024` : resolution native SDXL

In [ ]:
import os, hashlib, time

# ── Negative prompt — dit au modele ce qu'il doit EVITER ─────────────
# Critique pour eviter : deux nez, bouche deformee, visages fusionnes
NEGATIVE_PROMPT = (
    'deformed face, mutation, extra nose, double face, two faces, '
    'blurry, low quality, ugly, disfigured, bad anatomy, '
    'extra limbs, missing nose, cropped face, out of frame, '
    'watermark, text, jpeg artifacts, oversaturated'
)

def make_unique_seed(prompt: str) -> int:
    """
    Seed unique et non-reproductible : os.urandom + timestamp + prompt.
    Meme prompt = image differente a chaque appel.
    Sauvegarder le seed retourne pour reproduire une image specifique.
    """
    entropy = os.urandom(8).hex()
    raw     = f'{prompt}_{time.time_ns()}_{entropy}'
    return int(hashlib.sha256(raw.encode()).hexdigest(), 16) % (2**31)


def generate_image(pipe, prompt: str, seed: int = None,
                   guidance_scale: float = 9.0,
                   num_inference_steps: int = 50,
                   height: int = 1024, width: int = 1024,
                   negative_prompt: str = NEGATIVE_PROMPT):
    """
    Genere une image avec :
      - guidance_scale=9.0  : optimal SDXL portraits
      - negative_prompt     : evite deformations anatomiques
      - DPMSolverMultistep  : scheduler plus precis sur details faciaux
      - seed unique par defaut (reproductible si seed fourni)
    """
    from diffusers import DPMSolverMultistepScheduler

    # Remplacer le scheduler — DPMSolver donne des details plus nets
    # que le scheduler DDPM par defaut au meme nombre de steps
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config,
        use_karras_sigmas=True,
    )

    if seed is None:
        seed = make_unique_seed(prompt)

    g = torch.Generator(device='cuda').manual_seed(seed)
    with torch.no_grad():
        result = pipe(
            prompt,
            negative_prompt=negative_prompt,
            generator=g,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            height=height,
            width=width,
        )
    return result.images[0], seed


# ── 20 prompts EN uniquement (meilleure qualite sur SDXL) ────────────
# Les prompts FR donnent des resultats inferieurs car SDXL CLIP
# est principalement entraine sur de l'anglais.
TEST_PROMPTS = [
    'portrait photo of a young woman with brown hair smiling, highly detailed, 8k',
    'portrait photo of a young woman with blonde hair and blue eyes, sharp focus, 8k',
    'portrait photo of a young woman with black hair, studio lighting, 8k',
    'portrait photo of a young woman with red hair and freckles, photorealistic, 8k',
    'portrait photo of a young woman wearing glasses, professional photography, 8k',
    'portrait photo of a young man with dark hair and a beard, highly detailed, 8k',
    'portrait photo of a young man with blonde hair smiling, sharp focus, 8k',
    'portrait photo of a young man with brown hair, studio lighting, 8k',
    'portrait photo of a young man with glasses and short hair, photorealistic, 8k',
    'portrait photo of a young man with curly hair, professional photography, 8k',
    'portrait photo of an older woman with silver hair and a warm smile, 8k',
    'portrait photo of a mature woman with gray hair, highly detailed, 8k',
    'portrait photo of an older woman with white hair and kind eyes, 8k',
    'portrait photo of an older man with gray hair and a distinguished beard, 8k',
    'portrait photo of a mature man with white hair and a friendly smile, 8k',
    'portrait photo of a young woman with curly brown hair, warm smile, 8k',
    'portrait photo of a young man with black hair and strong jawline, 8k',
    'portrait photo of an elderly woman with wrinkles and wise eyes, 8k',
    'portrait photo of an elderly man with a kind expression, 8k',
    'portrait photo of a young woman with straight black hair and glasses, 8k',
]

print('=' * 70)
print(f'GENERATION DE {len(TEST_PROMPTS)} IMAGES')
print(f'  guidance_scale    = 9.0')
print(f'  negative_prompt   = actif (evite deformations anatomiques)')
print(f'  scheduler         = DPMSolverMultistep + Karras sigmas')
print(f'  prompts           = EN uniquement (meilleure qualite SDXL)')
print('=' * 70)

generated_images = []
generated_paths  = []
used_seeds       = []

for i, prompt in enumerate(TEST_PROMPTS):
    img, seed = generate_image(pipe, prompt)
    used_seeds.append(seed)
    path = SAMPLES_DIR / f'sample_{i:02d}_s{seed}.png'
    img.save(path)
    generated_images.append(img)
    generated_paths.append(path)
    print(f'[{i+1:2d}/{len(TEST_PROMPTS)}] seed={seed} | {prompt[:55]}...')

print(f'\nOK {len(generated_images)} images sauvegardees dans {SAMPLES_DIR}')

## 4. Visualisation des images generees

In [ ]:
# Grille 4x5 des images generees
fig = plt.figure(figsize=(20, 25))
gs = GridSpec(4, 5, figure=fig, hspace=0.45, wspace=0.25)
fig.suptitle('Images Generees par SDXL + LoRA CelebA', fontsize=16, fontweight='bold')

for idx, (img, prompt) in enumerate(zip(generated_images, TEST_PROMPTS)):
    ax = fig.add_subplot(gs[idx // 5, idx % 5])
    # Redimensionner pour affichage rapide
    display_img = img.resize((256, 256), Image.LANCZOS)
    ax.imshow(display_img)
    # Tronquer prompt long
    short = (prompt[:45] + '...') if len(prompt) > 45 else prompt
    ax.set_title(short, fontsize=7, wrap=True)
    ax.axis('off')

plt.tight_layout()
grid_path = WORKING_DIR / 'generated_samples_grid.png'
plt.savefig(grid_path, dpi=100, bbox_inches='tight')
print(f'Grille sauvegardee : {grid_path}')
plt.show()

## 5. CLIP Score — Alignement texte-image

Mesure a quel point les images generees correspondent aux prompts.
- **> 0.30** : Excellent
- **0.25–0.30** : Bon
- **0.20–0.25** : Acceptable
- **< 0.20** : Faible

In [ ]:
from evaluate import compute_clip_score

print('=' * 70)
print('CLIP SCORE — Alignement Texte-Image')
print('=' * 70)

clip_score = None
clip_scores_per_prompt = []

try:
    # Score global
    clip_score = compute_clip_score(generated_images, TEST_PROMPTS, device='cuda')

    # Scores individuels (un prompt a la fois)
    for img, prompt in zip(generated_images, TEST_PROMPTS):
        s = compute_clip_score([img], [prompt], device='cuda')
        clip_scores_per_prompt.append(s if s else 0.0)

    print(f'\nCLIP Score moyen : {clip_score:.4f}')
    print(f'Min              : {min(clip_scores_per_prompt):.4f}')
    print(f'Max              : {max(clip_scores_per_prompt):.4f}')
    print(f'Ecart-type       : {np.std(clip_scores_per_prompt):.4f}')

    if clip_score > 0.30:
        print(f'\nOK EXCELLENT — image/texte tres bien alignes')
    elif clip_score > 0.25:
        print(f'\nOK BON — alignement correct')
    elif clip_score > 0.20:
        print(f'\nACCEPTABLE — alignement partiel')
    else:
        print(f'\nFAIBLE — alignment insuffisant')

except Exception as e:
    print(f'Erreur CLIP : {e}')
    clip_score = None

print('=' * 70)

## 6. FID Score — Qualite vs images reelles CelebA

Le FID (Frechet Inception Distance) compare la distribution des images generees
a celle des vraies images CelebA.
- **< 10** : Excellent (quasi indiscernable)
- **10–30** : Tres bon
- **30–60** : Acceptable
- **> 60** : Faible

Note : avec seulement 20 images generees, le FID sera approximatif.
Un FID fiable necessite 500+ images. Cette valeur est indicative.

In [ ]:
from evaluate import extract_features, compute_fid

print('=' * 70)
print('FID SCORE — Qualite vs images reelles CelebA')
print('=' * 70)

fid_score = None

if not real_celeba_paths:
    print('SKIP : images CelebA reelles introuvables.')
    print('  Verifier que jessicali9530/celeba-dataset est ajoute au notebook.')
else:
    try:
        print(f'\nExtraction features images generees ({len(generated_paths)})...')
        gen_features = extract_features(generated_paths, device='cuda', batch_size=4)
        print(f'  Forme : {gen_features.shape}')

        print(f'\nExtraction features images reelles ({len(real_celeba_paths)})...')
        real_features = extract_features(real_celeba_paths, device='cuda', batch_size=32)
        print(f'  Forme : {real_features.shape}')

        print('\nCalcul FID...')
        fid_score = compute_fid(real_features, gen_features)

        print(f'\nFID Score : {fid_score:.2f}')
        qual = ('EXCELLENT'  if fid_score < 10 else
                'TRES BON'   if fid_score < 30 else
                'ACCEPTABLE' if fid_score < 60 else 'FAIBLE')
        print(f'Qualite   : {qual}')
        print(f'Note      : FID indicatif ({len(generated_paths)} images; fiable a partir de 500+)')

    except Exception as e:
        import traceback
        print(f'Erreur FID : {e}')
        traceback.print_exc()

print('=' * 70)


## 7. Dashboard de visualisation des metriques

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Dashboard Evaluation SDXL + LoRA CelebA', fontsize=14, fontweight='bold')

# ── Panel 1 : CLIP Score ────────────────────────────────────────────────
ax = axes[0, 0]
if clip_score:
    color = 'green' if clip_score > 0.25 else 'orange' if clip_score > 0.20 else 'red'
    ax.barh(['CLIP Score'], [clip_score], color=color, height=0.4)
    ax.set_xlim(0, 0.5)
    ax.axvspan(0.30, 0.5,  alpha=0.08, color='green')
    ax.axvspan(0.25, 0.30, alpha=0.08, color='blue')
    ax.axvspan(0.20, 0.25, alpha=0.08, color='yellow')
    ax.axvspan(0,    0.20, alpha=0.08, color='red')
    ax.text(clip_score + 0.005, 0, f'{clip_score:.4f}', va='center', fontweight='bold')
    ax.set_title('CLIP Score (Alignement Texte-Image)', fontweight='bold')
    patches = [
        mpatches.Patch(color='green',  alpha=0.4, label='Excellent (>0.30)'),
        mpatches.Patch(color='blue',   alpha=0.4, label='Bon (0.25-0.30)'),
        mpatches.Patch(color='yellow', alpha=0.4, label='Acceptable (0.20-0.25)'),
        mpatches.Patch(color='red',    alpha=0.4, label='Faible (<0.20)'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=8)
else:
    ax.text(0.5, 0.5, 'CLIP non disponible', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('CLIP Score', fontweight='bold')

# ── Panel 2 : FID Score ─────────────────────────────────────────────────
ax = axes[0, 1]
if fid_score is not None:
    color = 'green' if fid_score < 30 else 'orange' if fid_score < 60 else 'red'
    ax.bar(['FID Score'], [fid_score], color=color, width=0.4)
    ax.axhline(10, color='green',  linestyle='--', alpha=0.5, label='Excellent (<10)')
    ax.axhline(30, color='blue',   linestyle='--', alpha=0.5, label='Tres bon (<30)')
    ax.axhline(60, color='orange', linestyle='--', alpha=0.5, label='Acceptable (<60)')
    ax.set_title('FID Score (Qualite vs CelebA reelles)', fontweight='bold')
    ax.set_ylabel('FID (plus bas = meilleur)')
    ax.legend(fontsize=8)
    ax.text(0, fid_score + 1, f'{fid_score:.1f}', ha='center', fontweight='bold')
else:
    ax.text(0.5, 0.5, 'FID non calcule', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('FID Score', fontweight='bold')

# ── Panel 3 : CLIP par prompt ────────────────────────────────────────────
ax = axes[1, 0]
if clip_scores_per_prompt:
    colors = ['green' if s > 0.25 else 'orange' if s > 0.20 else 'red'
              for s in clip_scores_per_prompt]
    ax.bar(range(len(clip_scores_per_prompt)), clip_scores_per_prompt, color=colors)
    ax.axhline(0.25, color='green',  linestyle='--', alpha=0.5, label='Bon (0.25)')
    ax.axhline(0.20, color='orange', linestyle='--', alpha=0.5, label='Acceptable (0.20)')
    ax.set_xlabel('Prompt #')
    ax.set_ylabel('CLIP Score')
    ax.set_title('CLIP Score par Prompt', fontweight='bold')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'Pas de scores', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('CLIP Score par Prompt', fontweight='bold')

# ── Panel 4 : Resume config ──────────────────────────────────────────────
ax = axes[1, 1]
ax.axis('off')
clip_str = f'{clip_score:.4f}' if clip_score else 'N/A'
fid_str  = f'{fid_score:.1f}'  if fid_score  else 'N/A'
txt = (
    f'Configuration Evaluation\n\n'
    f'Modele     : SDXL + LoRA CelebA\n'
    f'Checkpoint : final (15000 steps)\n'
    f'Meilleure loss train : 0.023538\n\n'
    f'Generation\n'
    f'  Steps inference : 50\n'
    f'  Guidance scale  : 9.0\n'
    f'  Resolution      : 1024x1024\n'
    f'  Images generees : {len(generated_images)}\n\n'
    f'Metriques\n'
    f'  CLIP Score : {clip_str}\n'
    f'  FID Score  : {fid_str}\n'
)
ax.text(0.05, 0.95, txt, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
dashboard_path = WORKING_DIR / 'evaluation_dashboard.png'
plt.savefig(dashboard_path, dpi=100, bbox_inches='tight')
print(f'Dashboard sauvegarde : {dashboard_path}')
plt.show()

## 8. Rapport JSON et resume final

In [ ]:
from evaluate import EvaluationReport

print('=' * 70)
print('RAPPORT D EVALUATION FINAL')
print('=' * 70)

report = EvaluationReport()
if clip_score:
    report.add_metric('clip_score_mean', clip_score)
    report.add_metric('clip_score_min',  min(clip_scores_per_prompt) if clip_scores_per_prompt else None)
    report.add_metric('clip_score_max',  max(clip_scores_per_prompt) if clip_scores_per_prompt else None)
if fid_score is not None:
    report.add_metric('fid_score', fid_score)
    report.add_metric('fid_note', 'indicatif (20 images generees, fiable a partir de 500+)')

metadata = {
    'date_evaluation':     datetime.now().isoformat(),
    'checkpoint':          str(CHECKPOINT_PATH),
    'model_base':          MODEL_ID,
    'training_steps':      15000,
    'best_train_loss':     0.023538,
    'nombre_samples':      len(generated_images),
    'inference_steps':     50,
    'guidance_scale':      9.0,
    'resolution':          '1024x1024',
    'seeds_aleatoires':    True,
    'used_seeds':          used_seeds,
    'clip_score':          clip_score,
    'fid_score':           fid_score,
}

report_path   = WORKING_DIR / 'evaluation_report.json'
metadata_path = WORKING_DIR / 'evaluation_metadata.json'
report.save_report(report_path)
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(report.summary())
print(f'\nOK Rapport     : {report_path}')
print(f'OK Metadata    : {metadata_path}')
print(f'OK Images      : {SAMPLES_DIR}/ ({len(generated_images)} fichiers)')
print(f'OK Dashboard   : {WORKING_DIR}/evaluation_dashboard.png')
print(f'OK Grille      : {WORKING_DIR}/generated_samples_grid.png')

# Resume final
print('\n' + '=' * 70)
print('RESUME FINAL')
print('=' * 70)
print(f'Modele           : SDXL + LoRA CelebA (15 000 steps)')
print(f'Meilleure loss   : 0.023538 (excellente convergence)')
if clip_score:
    qual = 'EXCELLENT' if clip_score > 0.30 else 'BON' if clip_score > 0.25 else 'ACCEPTABLE' if clip_score > 0.20 else 'FAIBLE'
    print(f'CLIP Score       : {clip_score:.4f} ({qual})')
if fid_score is not None:
    qual_f = 'EXCELLENT' if fid_score < 10 else 'TRES BON' if fid_score < 30 else 'ACCEPTABLE' if fid_score < 60 else 'FAIBLE'
    print(f'FID Score        : {fid_score:.1f} ({qual_f}) [indicatif]')
print('=' * 70)